In [1]:
import numpy as np

stage = "stage2"
fname = "mass_ranges_stage2.txt"

train_file= "train_settings_NAME.yaml"
wave_file = "waveform_dataset_settings_NAME.yaml"
sh_file = "NAME.sh"

In [2]:
def chirp_mass(m1, m2):
    return (m1*m2)**(3/5) / (m1+m2)**(1/5)


def from_mChirp_q_to_m1_m2(mChirp, q):
    """
    Compute the transformation from mChirp, q to m1, m2
    """
    m1 = mChirp * (1 + q)**(1/5) * q**(-3/5)
    m2 = mChirp * (1 + q)**(1/5) * q**(2/5)
    return m1, m2

In [3]:
import pandas as pd

df = pd.read_csv(fname, sep=r"\s+", header=None, names=["CMIN", "CMAX"], dtype=float)

for index, row in df.iterrows():
    print (row["CMIN"], row["CMAX"])
    if row["CMIN"] < 50.0:
        row["CMIN"] = 50.0
    else:
        CMIN = row["CMIN"]
    if row ["CMAX"] > 1000.0:
        row["CMAX"] = 1000.0
    else:
        CMAX = row["CMAX"]
        CMIN = row["CMIN"]
    mchirp_vec = np.linspace(CMIN, CMAX, 200)
    mass_ratio_vec = np.linspace(0.125, 1.0, 200)
    mass1 = []
    mass2 = []
    for mass in mchirp_vec:
        for q in mass_ratio_vec:
            m1, m2 = from_mChirp_q_to_m1_m2(mass, q)
            mass1.append(m1)
            mass2.append(m2)

    df.loc[index, "MIN1"] = int(np.floor(np.min(mass1)))
    df.loc[index, "MAX1"] = int(np.ceil(np.max(mass1)))
    df.loc[index, "MIN2"] = int(np.floor(np.min(mass2)))
    df.loc[index, "MAX2"] = int(np.ceil(np.max(mass2)))
    df["NAME"] = df["CMIN"].map("{:g}".format) + "_" + df["CMAX"].map("{:g}".format)
    df["SAMPLES"] = ((df["CMAX"] - df["CMIN"]) / 75 * 1_000_000).round().astype(int)

df

25.0 100.0
50.0 125.0
75.0 150.0
100.0 175.0
125.0 200.0
125.0 275.0
175.0 325.0
225.0 375.0
275.0 425.0
325.0 475.0
375.0 525.0
400.0 625.0
475.0 700.0
525.0 825.0
600.0 975.0
750.0 1050.0
900.0 1050.0


,CMIN,CMAX,MIN1,MAX1,MIN2,MAX2,NAME,SAMPLES
0,50.0,100.0,57.0,357.0,22.0,115.0,50_100,666667
1,50.0,125.0,57.0,446.0,22.0,144.0,50_125,1000000
2,75.0,150.0,86.0,535.0,33.0,173.0,75_150,1000000
3,100.0,175.0,114.0,624.0,44.0,202.0,100_175,1000000
4,125.0,200.0,143.0,714.0,55.0,230.0,125_200,1000000
5,125.0,275.0,143.0,981.0,55.0,316.0,125_275,2000000
6,175.0,325.0,201.0,1159.0,77.0,374.0,175_325,2000000
7,225.0,375.0,258.0,1337.0,100.0,431.0,225_375,2000000
8,275.0,425.0,315.0,1516.0,122.0,489.0,275_425,2000000
9,325.0,475.0,373.0,1694.0,144.0,546.0,325_475,2000000


In [10]:
import os
import re


out_dir = stage
os.makedirs(out_dir, exist_ok=True)

with open(wave_file) as f:
    template = f.read()

for _, row in df.iterrows():
    values = {
        "NAME": row["NAME"],
        "MIN1": f"{row['MIN1']:g}",
        "MAX1": f"{row['MAX1']:g}",
        "MIN2": f"{row['MIN2']:g}",
        "MAX2": f"{row['MAX2']:g}",
        "CMIN": f"{row['CMIN']:g}",
        "CMAX": f"{row['CMAX']:g}",
        "SAMPLES": str(int(row["SAMPLES"])),
    }

    pattern = r"\b(" + "|".join(values) + r")\b"
    content = re.sub(pattern, lambda m: values[m.group(1)], template)

    out_file = os.path.join(out_dir, wave_file.replace("NAME", row["NAME"]))
    with open(out_file, "w") as f:
        f.write(content)
    print("written:", out_file)

written: stage2/waveform_dataset_settings_50_100.yaml
written: stage2/waveform_dataset_settings_50_125.yaml
written: stage2/waveform_dataset_settings_75_150.yaml
written: stage2/waveform_dataset_settings_100_175.yaml
written: stage2/waveform_dataset_settings_125_200.yaml
written: stage2/waveform_dataset_settings_125_275.yaml
written: stage2/waveform_dataset_settings_175_325.yaml
written: stage2/waveform_dataset_settings_225_375.yaml
written: stage2/waveform_dataset_settings_275_425.yaml
written: stage2/waveform_dataset_settings_325_475.yaml
written: stage2/waveform_dataset_settings_375_525.yaml
written: stage2/waveform_dataset_settings_400_625.yaml
written: stage2/waveform_dataset_settings_475_700.yaml
written: stage2/waveform_dataset_settings_525_825.yaml
written: stage2/waveform_dataset_settings_600_975.yaml
written: stage2/waveform_dataset_settings_750_1000.yaml
written: stage2/waveform_dataset_settings_900_1000.yaml


In [11]:
settings_tpl = "waveform_dataset_settings_NAME.yaml"
out_tpl = "/scratch/fs201368/tp41139/waveforms/waveform_dataset_NAME.hdf5"

for name in df["NAME"]:
    print(f"dingo_generate_dataset --settings {settings_tpl.replace('NAME', name)} "
          f"--out_file {out_tpl.replace('NAME', name)}")

dingo_generate_dataset --settings waveform_dataset_settings_50_100.yaml --out_file /scratch/fs201368/tp41139/waveforms/waveform_dataset_50_100.hdf5
dingo_generate_dataset --settings waveform_dataset_settings_50_125.yaml --out_file /scratch/fs201368/tp41139/waveforms/waveform_dataset_50_125.hdf5
dingo_generate_dataset --settings waveform_dataset_settings_75_150.yaml --out_file /scratch/fs201368/tp41139/waveforms/waveform_dataset_75_150.hdf5
dingo_generate_dataset --settings waveform_dataset_settings_100_175.yaml --out_file /scratch/fs201368/tp41139/waveforms/waveform_dataset_100_175.hdf5
dingo_generate_dataset --settings waveform_dataset_settings_125_200.yaml --out_file /scratch/fs201368/tp41139/waveforms/waveform_dataset_125_200.hdf5
dingo_generate_dataset --settings waveform_dataset_settings_125_275.yaml --out_file /scratch/fs201368/tp41139/waveforms/waveform_dataset_125_275.hdf5
dingo_generate_dataset --settings waveform_dataset_settings_175_325.yaml --out_file /scratch/fs201368/tp41

In [4]:
import os

train_file = "train_settings_NAME.yaml"
out_dir = "stage2"
os.makedirs(out_dir, exist_ok=True)

with open(train_file) as f:
    template = f.read()

for name in df["NAME"]:
    out_file = os.path.join(out_dir, train_file.replace("NAME", name))
    with open(out_file, "w") as f:
        f.write(template.replace("NAME", name))
    print("written:", out_file)

written: stage2/train_settings_50_100.yaml
written: stage2/train_settings_50_125.yaml
written: stage2/train_settings_75_150.yaml
written: stage2/train_settings_100_175.yaml
written: stage2/train_settings_125_200.yaml
written: stage2/train_settings_125_275.yaml
written: stage2/train_settings_175_325.yaml
written: stage2/train_settings_225_375.yaml
written: stage2/train_settings_275_425.yaml
written: stage2/train_settings_325_475.yaml
written: stage2/train_settings_375_525.yaml
written: stage2/train_settings_400_625.yaml
written: stage2/train_settings_475_700.yaml
written: stage2/train_settings_525_825.yaml
written: stage2/train_settings_600_975.yaml
written: stage2/train_settings_750_1000.yaml
written: stage2/train_settings_900_1000.yaml


In [5]:
import os
import re
import math

sh_file = "NAME.sh"
out_dir = "stage2"
os.makedirs(out_dir, exist_ok=True)

with open(sh_file) as f:
    template = f.read()

HOURS_PER_5M = 24   # 5_000_000 samples (range 375) -> 24 h

for _, row in df.iterrows():
    name = row["NAME"]

    # time proportional to dataset size, rounded up to full minutes
    minutes = math.ceil(row["SAMPLES"] / 5_000_000 * HOURS_PER_5M * 60)
    time_str = f"{minutes // 60}:{minutes % 60:02d}:00"

    content = template.replace("NAME", name)
    content = re.sub(r"(#SBATCH --time=).*", rf"\g<1>{time_str}", content)

    out_file = os.path.join(out_dir, sh_file.replace("NAME", name))
    with open(out_file, "w") as f:
        f.write(content)
    print(f"written: {out_file}  (time={time_str})")

written: stage2/50_100.sh  (time=3:13:00)
written: stage2/50_125.sh  (time=4:49:00)
written: stage2/75_150.sh  (time=4:49:00)
written: stage2/100_175.sh  (time=4:49:00)
written: stage2/125_200.sh  (time=4:49:00)
written: stage2/125_275.sh  (time=9:37:00)
written: stage2/175_325.sh  (time=9:37:00)
written: stage2/225_375.sh  (time=9:37:00)
written: stage2/275_425.sh  (time=9:37:00)
written: stage2/325_475.sh  (time=9:37:00)
written: stage2/375_525.sh  (time=9:37:00)
written: stage2/400_625.sh  (time=14:24:00)
written: stage2/475_700.sh  (time=14:24:00)
written: stage2/525_825.sh  (time=19:13:00)
written: stage2/600_975.sh  (time=24:00:00)
written: stage2/750_1000.sh  (time=16:00:00)
written: stage2/900_1000.sh  (time=6:24:00)


In [6]:
for name in df["NAME"]:
    print(f"sbatch {name}.sh")

sbatch 50_100.sh
sbatch 50_125.sh
sbatch 75_150.sh
sbatch 100_175.sh
sbatch 125_200.sh
sbatch 125_275.sh
sbatch 175_325.sh
sbatch 225_375.sh
sbatch 275_425.sh
sbatch 325_475.sh
sbatch 375_525.sh
sbatch 400_625.sh
sbatch 475_700.sh
sbatch 525_825.sh
sbatch 600_975.sh
sbatch 750_1000.sh
sbatch 900_1000.sh
